In [1]:
import kagglehub
from pycocotools.coco import COCO
import pandas as pd
import matplotlib.pyplot as plt

path = kagglehub.dataset_download(
    "awsaf49/coco-2017-dataset",
    "coco2017/annotations/instances_train2017.json"
)

# Initialize COCO API
coco = COCO(path)

loading annotations into memory...
Done (t=19.46s)
creating index...
index created!


In [2]:
# Get category IDs to fetch subsets
category_ids = coco.getCatIds(catNms=['person', 'car', 'bicycle', 'truck', 'bus'])

# Get image Ids for these categories
image_ids = coco.getImgIds(catIds=category_ids)

# Get annotation Ids for these categories
ann_ids = coco.getAnnIds(imgIds=image_ids, catIds=category_ids)

# Get categories
categories = coco.loadCats(category_ids)

In [3]:
# Load data into dataframes
images_df = pd.DataFrame(coco.loadImgs(image_ids)).rename(columns={'id': 'image_id'})
ann_df = pd.DataFrame(coco.loadAnns(ann_ids))
category_df = pd.DataFrame(categories).rename(columns={'id': 'category_id'})

In [4]:
# Link annotations to category names
merged = pd.merge(ann_df, category_df[['category_id', 'name', 'supercategory']], on='category_id')
# Link annotations to image metadata
final = pd.merge(merged, images_df, on='image_id')

final.head()

,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,"[[23.92, 277.81, 33.74, 277.29, 36.84, 277.12,...",256.16895,0,451594,"[19.09, 262.47, 20.33, 15.34]",3,345093,car,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
1,"[[223.39, 294.77, 268.58, 296.51, 287.12, 304....",20363.41525,0,451594,"[187.47, 178.33, 188.87, 126.29]",6,365984,bus,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
2,"[[87.07, 276.46, 83.81, 278.64, 75.1, 278.64, ...",4680.21885,0,451594,"[44.63, 228.57, 95.78, 62.04]",8,398502,truck,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
3,"[[437.47, 268.88, 440.66, 256.32, 444.09, 252....",503.78305,0,451594,"[436.79, 246.27, 18.83, 47.47]",1,426346,person,person,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
4,"[[464.8, 253.68, 464.28, 251.09, 463.63, 248.3...",413.69200,0,451594,"[460.13, 246.37, 12.97, 47.93]",1,452763,person,person,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...


In [5]:
final.shape

(1546, 16)

In [6]:
# Remove null rows
cleaned_df = final.dropna(how='any')
cleaned_df.shape

(1546, 16)

In [7]:
# Map COCO-original indices to 0-index for cleaner processing
map_ids = {old_id: index for index, old_id in enumerate(category_ids)}

# Create new column in dataframe to include 0-based indexing
cleaned_df['image_index'] = cleaned_df['category_id'].map(map_ids)

In [8]:
cleaned_df.head()

,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url,image_index
0,"[[23.92, 277.81, 33.74, 277.29, 36.84, 277.12,...",256.16895,0,451594,"[19.09, 262.47, 20.33, 15.34]",3,345093,car,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...,2
1,"[[223.39, 294.77, 268.58, 296.51, 287.12, 304....",20363.41525,0,451594,"[187.47, 178.33, 188.87, 126.29]",6,365984,bus,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...,3
2,"[[87.07, 276.46, 83.81, 278.64, 75.1, 278.64, ...",4680.21885,0,451594,"[44.63, 228.57, 95.78, 62.04]",8,398502,truck,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...,4
3,"[[437.47, 268.88, 440.66, 256.32, 444.09, 252....",503.78305,0,451594,"[436.79, 246.27, 18.83, 47.47]",1,426346,person,person,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...,0
4,"[[464.8, 253.68, 464.28, 251.09, 463.63, 248.3...",413.69200,0,451594,"[460.13, 246.37, 12.97, 47.93]",1,452763,person,person,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...,0


In [9]:
# Rearrange to make the index column to the 0th index

cols = cleaned_df.columns.tolist()
# Remove index column from current index and insert at 0th index
cols.insert(0, cols.pop(cols.index('image_index')))
# Reassign the dataframe with the new column order
cleaned_df = cleaned_df[cols]


In [10]:
cleaned_df.head()

,image_index,segmentation,area,iscrowd,image_id,bbox,category_id,id,name,supercategory,license,file_name,coco_url,height,width,date_captured,flickr_url
0,2,"[[23.92, 277.81, 33.74, 277.29, 36.84, 277.12,...",256.16895,0,451594,"[19.09, 262.47, 20.33, 15.34]",3,345093,car,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
1,3,"[[223.39, 294.77, 268.58, 296.51, 287.12, 304....",20363.41525,0,451594,"[187.47, 178.33, 188.87, 126.29]",6,365984,bus,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
2,4,"[[87.07, 276.46, 83.81, 278.64, 75.1, 278.64, ...",4680.21885,0,451594,"[44.63, 228.57, 95.78, 62.04]",8,398502,truck,vehicle,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
3,0,"[[437.47, 268.88, 440.66, 256.32, 444.09, 252....",503.78305,0,451594,"[436.79, 246.27, 18.83, 47.47]",1,426346,person,person,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...
4,0,"[[464.8, 253.68, 464.28, 251.09, 463.63, 248.3...",413.69200,0,451594,"[460.13, 246.37, 12.97, 47.93]",1,452763,person,person,4,000000451594.jpg,http://images.cocodataset.org/train2017/000000...,480,640,2013-11-16 18:21:46,http://farm9.staticflickr.com/8536/8643321887_...


In [11]:
# Convert dataframe to CSV and save to Kaggle's working directory
cleaned_df.to_csv('/kaggle/working/filtered_coco_metadata.csv', index=False)